# Model training for river flow forecasting

This notebook trains Random Forest, XGBoost, and ARIMA models to forecast
daily river flow (or level). We use a temporal train/test split (last 20%
of days as the test set) and evaluate with `TimeSeriesSplit` cross-validation.
All model functions come from `src/model.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

from src.data_loader import load_and_prepare
from src.model import (
    temporal_train_test_split, train_random_forest, train_xgboost,
    fit_arima, arima_forecast, evaluate, save_model,
)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load the daily feature dataset

In [ ]:
import os

daily_path = '../data/river_flow_daily_features.csv'
if os.path.exists(daily_path):
    daily_df = pd.read_csv(daily_path)
    print(f'Loaded daily features: {len(daily_df)} days')
else:
    daily_df = load_and_prepare(limit=50000)
    print(f'Built daily features: {len(daily_df)} days')

target_col = 'flow_rate' if 'flow_rate' in daily_df.columns else 'level'
print(f'Target: {target_col}')
daily_df.head()

## 2. Temporal train/test split

`temporal_train_test_split` from `model.py` respects chronological order:
the first 80% of days form the training set and the last 20% form the test
set. This prevents data leakage from future values.

In [ ]:
X_train, X_test, y_train, y_test = temporal_train_test_split(
    daily_df, target_col=target_col, test_fraction=0.20
)

print(f'Training set: {len(X_train)} days')
print(f'Test set:     {len(X_test)} days')
print(f'Features:     {list(X_train.columns)}')

## 3. Train Random Forest

In [ ]:
rf_model = train_random_forest(X_train, y_train, n_estimators=200, max_depth=15)
rf_preds = rf_model.predict(X_test)
rf_metrics = evaluate(y_test, rf_preds)

print('Random Forest metrics:')
for k, v in rf_metrics.items():
    print(f'  {k}: {v:.4f}')

## 4. Train XGBoost

In [ ]:
xgb_model = train_xgboost(X_train, y_train, n_estimators=300, max_depth=6, learning_rate=0.1)
xgb_preds = xgb_model.predict(X_test)
xgb_metrics = evaluate(y_test, xgb_preds)

print('XGBoost metrics:')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v:.4f}')

## 5. Fit ARIMA

`fit_arima` from `model.py` wraps `statsmodels.SARIMAX`. We use order (5,1,0)
based on the stationarity analysis from notebook 02: the first difference is
stationary (d=1), and 5 autoregressive terms capture the short-term persistence.

In [ ]:
# Build the full time series for ARIMA
daily_df['date'] = pd.to_datetime(daily_df['date'])
series = daily_df.set_index('date')[target_col]

split_idx = int(len(series) * 0.80)
train_series = series.iloc[:split_idx]
test_series = series.iloc[split_idx:]

arima_result = fit_arima(train_series, order=(5, 1, 0))

if arima_result is not None:
    print(f'ARIMA(5,1,0) AIC: {arima_result.aic:.2f}')
    forecast, lower, upper = arima_forecast(arima_result, steps=len(test_series))
    arima_metrics = evaluate(test_series.values, forecast)
    print('ARIMA metrics:')
    for k, v in arima_metrics.items():
        print(f'  {k}: {v:.4f}')
else:
    print('ARIMA fitting failed; skipping.')
    arima_metrics = None

## 6. Cross-validation with TimeSeriesSplit

Standard k-fold CV is not appropriate for time series because it would leak
future data into training folds. `TimeSeriesSplit` creates expanding-window
folds that respect temporal order.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

tscv = TimeSeriesSplit(n_splits=5)

# Full feature matrix (excluding date and target)
feature_cols = list(X_train.columns)
X_all = daily_df[feature_cols]
y_all = daily_df[target_col]

cv_models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=0),
}

cv_results = {}
for name, model in cv_models.items():
    mae_scores = -cross_val_score(model, X_all, y_all, cv=tscv, scoring='neg_mean_absolute_error')
    r2_scores = cross_val_score(model, X_all, y_all, cv=tscv, scoring='r2')
    cv_results[name] = {
        'CV MAE': f'{mae_scores.mean():.4f} +/- {mae_scores.std():.4f}',
        'CV R2': f'{r2_scores.mean():.4f} +/- {r2_scores.std():.4f}',
    }
    print(f'{name:16s}  MAE={mae_scores.mean():.4f} +/- {mae_scores.std():.4f}  '
          f'R2={r2_scores.mean():.4f} +/- {r2_scores.std():.4f}')

cv_df = pd.DataFrame(cv_results).T
cv_df

## 7. Model comparison table

In [ ]:
comparison = pd.DataFrame({
    'Random Forest': rf_metrics,
    'XGBoost': xgb_metrics,
})
if arima_metrics is not None:
    comparison['ARIMA(5,1,0)'] = arima_metrics

comparison = comparison.T.round(4)
comparison.index.name = 'Model'
print(comparison.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'R2']):
    comparison[metric].plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 8. Save the best model

In [ ]:
# Select best by R2
best_name = comparison['R2'].astype(float).idxmax()
print(f'Best model: {best_name} (R2 = {comparison.loc[best_name, "R2"]})')

if 'Random Forest' in best_name:
    save_model(rf_model, 'rf_river_flow.joblib')
elif 'XGBoost' in best_name:
    save_model(xgb_model, 'xgb_river_flow.joblib')
elif arima_result is not None:
    save_model(arima_result, 'arima_river_flow.joblib')

print('Model saved to ../models/')

## Summary

- Trained Random Forest and XGBoost on lag-based and rolling-average features
  using a temporal 80/20 split.
- Fit an ARIMA(5,1,0) model on the univariate daily flow series.
- TimeSeriesSplit cross-validation (5 folds) gives robust estimates that
  account for temporal dependence.
- ML models (especially XGBoost) tend to outperform ARIMA on multi-step
  forecasts because they can exploit calendar and rolling features.
- The best model was saved for detailed evaluation in notebook 04.